# Breast Cancer Classification using Logistic Regression and Neural Networks

This notebook combines the code used for the breast cancer classification project into one place.

The notebook includes:

1. Logistic regression baseline
2. Baseline neural network
3. Neural network with weight decay
4. 5-fold cross-validation with weight decay

The only code change made during consolidation is adding `stratify=y` to the train/test splits so the class balance is preserved in the training and test sets.

## Logistic Regression Baseline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()


X = data.data
y = torch.from_numpy(data.target).type(torch.float)

In [ ]:
#create train/test split of data
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)


#set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

#scale data
scaler = StandardScaler()

X_train = torch.from_numpy(scaler.fit_transform(X_train)).type(torch.float).to(device)
X_test = torch.from_numpy(scaler.transform(X_test)).type(torch.float).to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)
X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [ ]:
#building a model

class RegressionModel(nn.Module):
    def __init__(self):
        super().__init__()

        #create input layer with 30 input features to match shape of data
        self.input_layer = nn.Linear(in_features=30, out_features=1)

    def forward(self, x):
        logits = self.input_layer(x)
        return logits

In [ ]:
# create random seed
torch.manual_seed(42)

# create instance of model
model_1 = RegressionModel()

#check model's parameters
list(model_1.parameters())

#list names parameters
model_1.state_dict()

In [ ]:
# set up loss function and optimizer

loss_fn = nn.BCEWithLogitsLoss() #BCE with sigmoid built in

optimizer = torch.optim.SGD(model_1.parameters(), lr=0.01) # stochastic gradient descent

#calculate accuracy
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = correct/len(y_pred)*100
  return acc

In [ ]:
#building training and testing loop

epoch_count = []
loss_values = []
test_loss_values = []
accuracy = []
test_accuracy = []

torch.manual_seed(42)

#set number of epochs
epochs = 60

#training loop
for epoch in range(epochs):

  #activate training mode
  model_1.train()

  #forward pass
  y_logits = model_1(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits)) # logits -> pred probs -> pred labels

  #calculate loss/accuracy
  loss = loss_fn(y_logits, #loss function expects raw logits as input
                y_train)
  acc = accuracy_fn(y_true=y_train,
                   y_pred=y_pred)

  #optimizer 0 grad
  optimizer.zero_grad()

  #loss backward
  loss.backward()

  #optimizer step
  optimizer.step()

  #testing
  model_1.eval()
  with torch.inference_mode():
    #forward pass
    test_logits = model_1(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    #calculate test loss /acc
    test_loss = loss_fn(test_logits,
                       y_test)
    test_acc = accuracy_fn(y_true=y_test,
                          y_pred=test_pred)

  if epoch % 2 == 0:
    epoch_count.append(epoch)
    loss_values.append(loss.detach().numpy())
    test_loss_values.append(test_loss.detach().numpy())
    accuracy.append(acc)
    test_accuracy.append(test_acc)
    print(f"Epoch: {epoch} | Loss: {loss} | Test Loss: {test_loss} | Accuracy: {acc} | Test Accuracy: {test_acc}")

In [ ]:
plt.figure()

plt.plot(epoch_count, loss_values, label="Train Loss")
plt.plot(epoch_count, test_loss_values, label="Test Loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Test Loss")
plt.legend()

plt.figure()

plt.plot(epoch_count, accuracy, label="Train Accuracy")
plt.plot(epoch_count, test_accuracy, label="Test Accuracy")

plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Test Accuracy")
plt.legend()

plt.show()

## Baseline Neural Network

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()

X = data.data
y = torch.from_numpy(data.target).type(torch.float)

In [ ]:
#shape of data
print(X.shape)
print(y.shape)
unique, counts = np.unique(y, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
#create train/test split of data
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)


#set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

#scale data
scaler = StandardScaler()

X_train = torch.from_numpy(scaler.fit_transform(X_train)).type(torch.float).to(device)
X_test = torch.from_numpy(scaler.transform(X_test)).type(torch.float).to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)
X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [ ]:
#building a model

class BreastCancerModel(nn.Module):
    def __init__(self):
        super().__init__()

        #introduce the input layer with 30 features to match the shape of data
        self.layer_1 = nn.Linear(in_features=30, out_features=16)
        #add hidden reLu layer to introduce non-linearity
        self.hidden_layer = nn.ReLU()
        #output layer that has 1 out feature being either being between 0 and 1
        self.output_layer = nn.Linear(in_features=16, out_features=1)

        #define a forward method that outlines the forward pass
    def forward(self, x):
        x = self.layer_1(x)
        x = self.hidden_layer(x)
        x = self.output_layer(x)
        return x

#create an instance of our model
model_0 = BreastCancerModel().to(device)


In [ ]:
#create some untrained predictions

with torch.inference_mode():
  untrained_preds = model_0(X_test.to(device))
  probs = torch.sigmoid(untrained_preds)

torch.round(probs)[:10], y_test[:10]

In [ ]:
#setting loss function and optimizer

#use torch.nn.BECWithLogits
loss_fn  = nn.BCEWithLogitsLoss() # BCE loss with sigmoid activation built in

#use adam as optimizer
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.01)

#calculate accuracy
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = correct/len(y_pred)*100
  return acc




In [ ]:
#building training and testing loop

epoch_count = []
loss_values = []
test_loss_values = []
accuracy = []
test_accuracy = []

torch.manual_seed(42)

#set number of epochs
epochs = 60

#training loop
for epoch in range(epochs):

  #activate training mode
  model_0.train()

  #forward pass
  y_logits = model_0(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits)) # logits -> pred probs -> pred labels

  #calculate loss/accuracy
  loss = loss_fn(y_logits, #loss function expects raw logits as input
                y_train)
  acc = accuracy_fn(y_true=y_train,
                   y_pred=y_pred)

  #optimizer 0 grad
  optimizer.zero_grad()

  #loss backward
  loss.backward()

  #optimizer step
  optimizer.step()

  #testing
  model_0.eval()
  with torch.inference_mode():
    #forward pass
    test_logits = model_0(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    #calculate test loss /acc
    test_loss = loss_fn(test_logits,
                       y_test)
    test_acc = accuracy_fn(y_true=y_test,
                          y_pred=test_pred)

  if epoch % 2 == 0:
    epoch_count.append(epoch)
    loss_values.append(loss.detach().numpy())
    test_loss_values.append(test_loss.detach().numpy())
    accuracy.append(acc)
    test_accuracy.append(test_acc)
    print(f"Epoch: {epoch} | Loss: {loss} | Test Loss: {test_loss} | Accuracy: {acc} | Test Accuracy: {test_acc}")




In [ ]:
plt.figure()

plt.plot(epoch_count, loss_values, label="Train Loss")
plt.plot(epoch_count, test_loss_values, label="Test Loss")

plt.yticks(np.arange(0, 0.8, 0.05))

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Test Loss")
plt.legend()


plt.figure()

plt.plot(epoch_count, accuracy, label="Train Accuracy")
plt.plot(epoch_count, test_accuracy, label="Test Accuracy")

plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Test Accuracy")
plt.legend()

plt.show()

## Neural Network with Weight Decay

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

data = load_breast_cancer()

X = data.data
y = torch.from_numpy(data.target).type(torch.float)

#set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
#create train/test split of data
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)


#set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

#scale data
scaler = StandardScaler()

X_train = torch.from_numpy(scaler.fit_transform(X_train)).type(torch.float).to(device)
X_test = torch.from_numpy(scaler.transform(X_test)).type(torch.float).to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
#building a model

class BreastCancerModel(nn.Module):
    def __init__(self):
        super().__init__()

        #introduce the input layer with 30 features to match the shape of data
        self.layer_1 = nn.Linear(in_features=30, out_features=16)
        #add hidden reLu layer to introduce non-linearity
        self.hidden_layer = nn.ReLU()
        #output layer that has 1 out feature being either being between 0 and 1
        self.output_layer = nn.Linear(in_features=16, out_features=1)

        #define a forward method that outlines the forward pass
    def forward(self, x):
        x = self.layer_1(x)
        x = self.hidden_layer(x)
        x = self.output_layer(x)
        return x

#create an instance of our model
model_0 = BreastCancerModel().to(device)

In [ ]:
#setting loss function and optimizer

#use torch.nn.BECWithLogits
loss_fn  = nn.BCEWithLogitsLoss() # BCE loss with sigmoid activation built in

#use adam as optimizer
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.01, weight_decay = 0.002)

#calculate accuracy
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = correct/len(y_pred)*100
  return acc

In [ ]:
#building training and testing loop

epoch_count = []
loss_values = []
test_loss_values = []
accuracy = []
test_accuracy = []

torch.manual_seed(42)

#set number of epochs
epochs = 60

#training loop
for epoch in range(epochs):

  #activate training mode
  model_0.train()

  #forward pass
  y_logits = model_0(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits)) #logits -> pred probs -> pred labels

  #calculate loss/accuracy
  loss = loss_fn(y_logits, #loss function expects raw logits as input
                y_train)
  acc = accuracy_fn(y_true=y_train,
                   y_pred=y_pred)

  #optimizer 0 grad
  optimizer.zero_grad()

  #loss backward
  loss.backward()

  #optimizer step
  optimizer.step()

  #testing
  model_0.eval()
  with torch.inference_mode():
    #forward pass
    test_logits = model_0(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    #calculate test loss /acc
    test_loss = loss_fn(test_logits,
                       y_test)
    test_acc = accuracy_fn(y_true=y_test,
                          y_pred=test_pred)

  if epoch % 2 == 0:
    epoch_count.append(epoch)
    loss_values.append(loss.detach().numpy())
    test_loss_values.append(test_loss.detach().numpy())
    accuracy.append(acc)
    test_accuracy.append(test_acc)
    print(f"Epoch: {epoch} | Loss: {loss} | Test Loss: {test_loss} | Accuracy: {acc} | Test Accuracy: {test_acc}")




In [ ]:
plt.figure()

plt.plot(epoch_count, loss_values, label="Train Loss")
plt.plot(epoch_count, test_loss_values, label="Test Loss")

plt.yticks(np.arange(0, 0.8, 0.05))

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Test Loss")
plt.legend()


plt.figure()

plt.plot(epoch_count, accuracy, label="Train Accuracy")
plt.plot(epoch_count, test_accuracy, label="Test Accuracy")

plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Test Accuracy")
plt.legend()

plt.show()

## Cross-Validation with Weight Decay

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

data = load_breast_cancer()

X = data.data
y = torch.from_numpy(data.target).type(torch.float)

#set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
#building a model

class BreastCancerModel(nn.Module):
    def __init__(self):
        super().__init__()

        #introduce the input layer with 30 features to match the shape of data
        self.layer_1 = nn.Linear(in_features=30, out_features=16)
        #add hidden reLu layer to introduce non-linearity
        self.hidden_layer = nn.ReLU()
        #output layer that has 1 out feature being either being between 0 and 1
        self.output_layer = nn.Linear(in_features=16, out_features=1)

        #define a forward method that outlines the forward pass
    def forward(self, x):
        x = self.layer_1(x)
        x = self.hidden_layer(x)
        x = self.output_layer(x)
        return x

In [ ]:
#calculate accuracy
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = correct/len(y_pred)*100
  return acc

In [ ]:
#impliment cross validation

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#create vectors to hold fold accuracy/loss
fold_accuracies = []
fold_train_accuracies = []
fold_final_train_losses = []
fold_final_test_losses = []


#initialize loop
for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y.cpu().numpy())):

  #train and test splits for each fold
  X_train = X[train_idx]
  X_test = X[test_idx]
  y_train = y[train_idx]
  y_test = y[test_idx]

  #scale values
  scaler = StandardScaler()
  scaler.fit(X_train)  #learn μ, σ from training data
  X_train = scaler.transform(X_train)
  X_test = scaler.transform(X_test)  #use SAME μ, σ

  #convert data to tensors and send to device
  X_train = torch.from_numpy(X_train).type(torch.float).to(device)
  X_test = torch.from_numpy(X_test).type(torch.float).to(device)
  y_train = y_train.to(device)
  y_test = y_test.to(device)

  #create instance of model in loop
  model_0 = BreastCancerModel().to(device)

  #setting loss function and optimizer

  #use torch.nn.BECWithLogits
  loss_fn  = nn.BCEWithLogitsLoss() # BCE loss with sigmoid activation built in

  #use adam as optimizer
  optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.01, weight_decay=0.002) #add weight decay parameter to avoid model learning large weights

  #keep track of losses, accuracies and epoch count
  test_losses = []
  train_losses= []
  accuracy = []
  test_accuracy = []

  epoch_count = []

  #set number of epochs
  epochs = 60

  #training loop
  for epoch in range(epochs):

    #activate training mode
    model_0.train()

    #forward pass
    y_logits = model_0(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits)) # logits -> pred probs -> pred labels

    #calculate loss/accuracy
    loss = loss_fn(y_logits, #loss function expects raw logits as input
                  y_train)
    acc = accuracy_fn(y_true=y_train,
                    y_pred=y_pred)

    #optimizer 0 grad
    optimizer.zero_grad()

    #loss backward
    loss.backward()

    #optimizer step
    optimizer.step()

    #testing
    model_0.eval()
    with torch.inference_mode():
      #forward pass
      test_logits = model_0(X_test).squeeze()
      test_pred = torch.round(torch.sigmoid(test_logits))

      #calculate test loss /acc
      test_loss = loss_fn(test_logits,
                        y_test)
      test_acc = accuracy_fn(y_true=y_test,
                            y_pred=test_pred)

    if epoch % 5 == 0:
      epoch_count.append(epoch)
      train_losses.append(loss.detach().numpy())
      test_losses.append(test_loss.detach().numpy())
      accuracy.append(acc)
      test_accuracy.append(test_acc)

  #append fold loss values/accuracy
  fold_final_train_losses.append(loss.item())
  fold_final_test_losses.append(test_loss.item())
  fold_accuracies.append(test_acc)
  fold_train_accuracies.append(acc)

print("\n===== Cross Validation Results =====")
print(f"Fold Train Losses: {fold_final_train_losses}")
print(f"Fold Test Losses: {fold_final_test_losses}")
print(f"Fold Accuracies: {fold_accuracies}")
print(f"Fold Train Accuracies: {fold_train_accuracies}")
print(f"Mean Accuracy: {np.mean(fold_accuracies):.2f}%")
print(f"Std Dev (Accuracy): {np.std(fold_accuracies):.2f}")
print(f"Mean Train Accuracy: {np.mean(fold_train_accuracies):.2f}%")
print(f"Std Dev (Train Accuracy): {np.std(fold_train_accuracies):.2f}")
print(f"Mean Train loss: {100*np.mean(fold_final_train_losses):.2f}%")
print(f"Std Dev (Train Loss): {np.std(fold_final_train_losses):.2f}")
print(f"Mean Test loss: {100*np.mean(fold_final_test_losses):.2f}%")
print(f"Std Dev (Test Loss): {np.std(fold_final_test_losses):.2f}")









